# Getting started with tes-client

This notebook walks you through everything you need to submit and monitor computational tasks on a [GA4GH TES](https://ga4gh.github.io/task-execution-schemas/) endpoint using this library.

**What is TES?**

TES (Task Execution Service) is a standard REST API designed to let you run containerised jobs on distributed compute infrastructure — HPC clusters, cloud platforms, or local servers — without needing to know the details of the underlying scheduler. You describe *what* you want to run (a Docker image and a command), *what data* goes in and out, and *what resources* you need. TES handles the rest.

**What does this library do?**

It gives you a clean Python interface to:
- Authenticate with a Keycloak-secured TES endpoint (or skip auth entirely for open endpoints)
- Build a TES task using a simple, readable Python API
- Submit the task and get back an ID
- Track it through to completion, with automatic polling and state-change logging

---

## Contents

1. [Installation](#1-installation)
2. [Authentication](#2-authentication)
3. [Building a task](#3-building-a-task)
4. [Connecting to TES](#4-connecting-to-tes)
5. [Submitting a task](#5-submitting-a-task)
6. [Tracking to completion](#6-tracking-to-completion)
7. [Checking and refreshing auth in scripts](#7-checking-and-refreshing-auth-in-scripts)
8. [Other useful operations](#8-other-useful-operations)
9. [Handling failures](#9-handling-failures)

## 1. Installation

Install the library with pip. You only need to do this once (or whenever you update).

In [ ]:
# Run this cell to install the library
# If you're working in a virtual environment, make sure it's active first
%pip install tes-client

## 2. Authentication

Before you can talk to a TES endpoint, you need to prove who you are. This library supports four authentication methods. They all produce the same result — an `auth` object — so you can swap between them without changing any other code.

---

### Option A: No authentication

Use this for local development endpoints, or any TES server that is open to all users. No credentials needed.

In [ ]:
from tes_client import NoAuth

auth = NoAuth()

print("Auth object created:", auth)
print("Auth header it sends:", auth.auth_header())  # empty dict — no token needed

---

### Option B: Client credentials (machine-to-machine)

This is the right choice for automated scripts and pipelines. You register a *service account* in Keycloak and give it a client ID and secret. No browser, no user interaction.

The library fetches a token automatically the first time it's needed, and fetches a fresh one whenever the current token expires — you never have to manage tokens yourself.

In [ ]:
# Uncomment and fill in your real values to use this method

# from tes_client import ClientCredentialsAuth
#
# auth = ClientCredentialsAuth(
#     base_url="https://keycloak.example.org",   # your Keycloak server URL
#     realm="my-realm",                           # the Keycloak realm name
#     client_id="tes-service-account",            # the client ID registered in Keycloak
#     client_secret="super-secret",               # the client secret from Keycloak
# )

---

### Option C: Username and password

Sometimes called the ROPC (Resource Owner Password Credentials) grant. Good for interactive use when you don't want a browser pop-up. The library fetches an initial token with your credentials and then silently uses the refresh token for subsequent calls, so you only enter your password once per session.

> **Note:** ROPC must be explicitly enabled on the Keycloak realm. Ask your administrator if you're not sure.

In [ ]:
# Uncomment and fill in your real values to use this method

# from tes_client import PasswordAuth
#
# auth = PasswordAuth(
#     base_url="https://keycloak.example.org",
#     realm="my-realm",
#     client_id="tes-client",
#     username="alice",
#     password="hunter2",
#     client_secret="optional-only-if-confidential-client",  # leave out if not needed
# )

---

### Option D: Authorization code + PKCE (browser login)

The most secure option for interactive sessions. The first time you use it, a browser window opens and you log in normally through Keycloak. After that, the refresh token is reused silently until it expires, at which point the browser opens again.

This uses the PKCE extension (RFC 7636), which means no client secret is needed — it works with *public clients* registered in Keycloak.

In [ ]:
# Uncomment and fill in your real values to use this method

# from tes_client import AuthorizationCodeAuth
#
# auth = AuthorizationCodeAuth(
#     base_url="https://keycloak.example.org",
#     realm="my-realm",
#     client_id="tes-public-client",  # must be a public client in Keycloak
#     redirect_port=8080,             # must match the redirect URI configured in Keycloak
# )
#
# # The browser will open the first time you make a request — not here.
# # You can also trigger it immediately:
# # auth.ensure_valid()

---

**For the rest of this notebook we'll use `NoAuth`**, which works with any open endpoint. To use a real authenticated endpoint, replace the `auth = NoAuth()` line above with one of the other options and re-run.

## 3. Building a task

A TES task describes *what you want to run*. It has five main parts:

| Part | What it is |
|---|---|
| **Executors** | The Docker containers to run, and the command to run in each |
| **Inputs** | Files to download into the container before it starts |
| **Outputs** | Files to upload from the container after it finishes |
| **Resources** | How many CPUs, how much RAM and disk the job needs |
| **Tags** | Arbitrary metadata — useful for routing to a compute queue or tagging with a project name |

Tasks are built using a *fluent* (method-chaining) API. Each builder method returns the task object itself, so you can chain calls together into one readable block.

---

### Minimal task: just run a command

The simplest possible task — one executor, no file staging.

In [ ]:
from tes_client import TesTask

task = (
    TesTask(name="hello-world")
    .add_executor(
        image="ubuntu",           # any Docker image
        command=["echo", "Hello from TES!"],
    )
)

# Print the JSON that would be sent to the TES API
# This is useful for checking your task looks right before submitting
print(task.submission_json())

---

### Full task: inputs, outputs, and resources

A more realistic task that stages data in, runs a tool, and stages results out. We also request specific compute resources and attach tags so the scheduler can route it to the right queue.

In [ ]:
task = (
    TesTask(name="samtools-view")

    # Tags are key/value metadata.
    # special helper method for 5 safes TES platform
    # set_project_tag sets tags["project"] — used to associate this job with a project.
    # set_tres_tag sets tags["tres"] — used by some schedulers to select a compute queue.
    .set_project_tag("DPUK_Project55")
    .set_tres_tag("DPUK")

    # Inputs: files to pull into the container before it runs.
    # `url` is where to fetch the file from; `path` is where it appears inside the container.
    .add_input(
        url="s3://my-bucket/data/sample.bam",
        path="/inputs/sample.bam",
    )

    # Outputs: files to push out of the container after it finishes.
    # `path` is the file location inside the container; `url` is the destination.
    .add_output(
        path="/outputs/sample.flagstat",
        url="s3://my-bucket/results/sample.flagstat",
    )

    # Resources: ask for 2 CPU cores, 4 GB RAM, 20 GB disk.
    # Omit any you don't care about — the TES server uses its defaults.
    .set_resources(cpu_cores=2, ram_gb=4, disk_gb=20)

    # Executor: the Docker image and command to run.
    # The command sees the inputs and outputs at the paths you specified above.
    .add_executor(
        image="biocontainers/samtools:1.18",
        command=["samtools", "flagstat", "/inputs/sample.bam"],
        stdout="/outputs/sample.flagstat",  # capture stdout to this file
    )
)

print(task.submission_json())

Notice how neatly the JSON maps to what you described in Python. This is the message that gets sent to the TES API when you call `client.submit(task)`.

---

### Multiple executors

TES supports running more than one container *sequentially* within a single task. Each executor runs in order; if one fails, the task stops. This is useful for pipelines with a fixed pre-processing step.

In [ ]:
task_multi = (
    TesTask(name="index-then-view")
    .add_input(url="s3://my-bucket/data/sample.bam", path="/inputs/sample.bam")
    .add_output(path="/outputs/sample.bai", url="s3://my-bucket/results/sample.bai")

    # Step 1: index the BAM file
    .add_executor(
        image="biocontainers/samtools:1.18",
        command=["samtools", "index", "/inputs/sample.bam", "/outputs/sample.bai"],
    )

    # Step 2: a second container runs only after step 1 completes successfully
    .add_executor(
        image="ubuntu",
        command=["echo", "Indexing done!"],
    )
)

print(task_multi.submission_json())

## 4. Connecting to TES

`TesClient` is the object that talks to the TES server. You give it the server URL and the auth object you created earlier. All requests go through this client.

In [ ]:
from tes_client import TesClient, NoAuth

# Replace this URL with your actual TES server
TES_URL = "http://192.168.99.53:8000"

auth = NoAuth()   # swap this for ClientCredentialsAuth etc. as needed

client = TesClient(
    tes_url=TES_URL,
    token_manager=auth,
    timeout=60.0,    # seconds to wait for a response (default is 60)
)

print("Client created. Endpoint:", TES_URL)

### Check the server is reachable

`service_info()` calls the TES `/v1/service-info` endpoint, which returns metadata about the server — its name, version, and what features it supports. It's a handy way to confirm everything is working before you submit anything.

In [ ]:
import json

try:
    info = client.service_info()
    print(json.dumps(info, indent=2))
except Exception as e:
    print(f"Could not reach TES server: {e}")
    print("Make sure TES_URL is correct and the server is running.")

## 5. Submitting a task

`client.submit(task)` sends the task to the TES server and immediately returns a **task ID** — a unique string the server assigns to this run. The task is now queued on the server; your Python process is free to do other things while it runs.

Keep the task ID — you need it to track progress, fetch logs, or cancel the task.

In [ ]:
from tes_client import TesTask

# Build a simple task for this example
task = (
    TesTask(name="notebook-example")
    .set_project_tag("DPUK_Project55")
    .set_tres_tag("DPUK")
    .add_executor(
        image="ubuntu",
        command=["echo", "Hello from TES!"],
    )
)

# Submit — this is the line that actually sends the HTTP request
task_id = client.submit(task)

print(f"Task submitted!")
print(f"Task ID: {task_id}")
print()
print("Copy this ID — you'll use it to check status and fetch logs.")

You can check the current state immediately after submitting — it will usually be `QUEUED` at this point:

In [ ]:
# Quick state check — just asks for the current state, no waiting
current_state = client.state(task_id)
print(f"Current state: {current_state}")

## 6. Tracking to completion

`TaskTracker` handles the polling loop for you. You give it the client, the task ID, and how often to check; it prints each state transition and blocks until the task reaches a terminal state.

**Terminal states** — states where the task has finished (one way or another):

| State | Meaning |
|---|---|
| `COMPLETE` | All executors finished with exit code 0 |
| `EXECUTOR_ERROR` | An executor exited with a non-zero code |
| `SYSTEM_ERROR` | Infrastructure-level failure (node crash, out of disk, etc.) |
| `CANCELED` | Canceled by a call to `client.cancel()` |

In [ ]:
from tes_client import TaskTracker

tracker = TaskTracker(
    client,
    task_id,
    poll_interval=15.0,   # check every 15 seconds (adjust to suit your task duration)
)

# wait() blocks until COMPLETE, EXECUTOR_ERROR, SYSTEM_ERROR, or CANCELED.
# It raises TimeoutError if the task hasn't finished within `timeout` seconds.
final = tracker.wait(timeout=3600)  # give up after 1 hour

print(f"\nFinal state: {final.state}")

### Reacting to the outcome

In [ ]:
if final.state.is_failure():
    print(f"Task did not complete successfully (state: {final.state})")

    # tracker.report() prints a detailed summary: executor logs, exit codes,
    # stderr output, and any system log messages from the TES server
    tracker.report()

else:
    print("Task completed successfully!")

### Custom state-change callback

If you want to do something custom on every state transition — send a notification, write to a log file, update a database — pass a callback function:

In [ ]:
import datetime

def on_state_change(task):
    """Called every time the task moves to a new state."""
    timestamp = datetime.datetime.now().strftime("%H:%M:%S")
    print(f"  [{timestamp}] Task {task.id[:8]}… moved to {task.state}")
    # You could also: send a Slack message, write to a DB, trigger another job, etc.

tracker_with_callback = TaskTracker(
    client,
    task_id,
    poll_interval=15.0,
    on_state_change=on_state_change,
)

# final = tracker_with_callback.wait(timeout=3600)

## 7. Checking and refreshing auth in scripts

Tokens have a limited lifetime — typically 5–60 minutes depending on your Keycloak configuration. If you're running a long script that submits many tasks in a loop, the token you fetched at the start may expire before you're done.

The library handles this automatically when you call `client.submit()` or any other method — it checks the token before every request and refreshes it if needed. But sometimes it's useful to check or force a refresh *explicitly*, for example to:

- Print a warning before a long batch run
- Proactively refresh before a tight deadline
- Verify credentials are still valid at the start of a script

Every auth class provides two methods for this:

| Method | What it does |
|---|---|
| `auth.is_valid()` | Returns `True` if the cached token is still usable — **no network call** |
| `auth.ensure_valid()` | Refreshes or re-authenticates **now** if needed; no-op if the token is still good |

In [ ]:
# Check whether the current token is still usable — this is instant, no network call
if auth.is_valid():
    print("Token is valid — good to go.")
else:
    print("Token has expired or was never fetched.")

# Force a refresh if needed — safe to call at any time
auth.ensure_valid()
print("Auth is now guaranteed to be valid.")

### Pattern: long-running batch script

A typical pattern for a script that submits many tasks:

In [ ]:
# Simulated list of items to process
work_items = ["sample_A", "sample_B", "sample_C"]

for sample_name in work_items:

    # Proactively refresh auth before each submission.
    # If the token is still valid this is instant.
    # If it has expired, the library fetches a new one now rather than
    # letting the submit call fail mid-flight.
    if not auth.is_valid():
        print(f"Token expired before processing {sample_name} — re-authenticating…")
    auth.ensure_valid()

    task = (
        TesTask(name=f"process-{sample_name}")
        .add_executor(image="ubuntu", command=["echo", f"Processing {sample_name}"])
    )

    submitted_id = client.submit(task)
    print(f"  Submitted {sample_name} → {submitted_id}")

## 8. Other useful operations

### Fetch a task by ID

You can fetch a task at any time using its ID. The default `MINIMAL` view returns state and basic metadata. Pass `full=True` to get executor logs and stderr output — useful for debugging.

In [ ]:
# Minimal view — fast, just the state and name
task_minimal = client.get(task_id)
print(f"State: {task_minimal.state}")
print(f"Name:  {task_minimal.name}")

print()

# Full view — includes executor logs (stdout, stderr, exit code)
task_full = client.get(task_id, full=True)
if task_full.logs:
    for attempt in task_full.logs:
        for executor_log in attempt.logs:
            print(f"Exit code : {executor_log.exit_code}")
            if executor_log.stderr:
                print(f"Stderr    : {executor_log.stderr}")

### List tasks

`list_tasks()` returns a page of tasks from the server. You can filter by name prefix and control page size.

In [ ]:
# List the most recent 10 tasks
tasks = client.list_tasks(page_size=10)
print(f"Found {len(tasks)} tasks:")
for t in tasks:
    print(f"  {t.id}  {t.state}  {t.name or '(unnamed)'}")

In [ ]:
# Filter by name prefix — useful for finding tasks from a specific project or run
my_tasks = client.list_tasks(name_prefix="notebook-", page_size=20)
print(f"Found {len(my_tasks)} tasks starting with 'notebook-'")
for t in my_tasks:
    print(f"  {t.id}  {t.state}  {t.name}")

### Cancel a running task

If you submit a task and then realise you no longer need it, you can request cancellation. The task moves to `CANCELING` and then `CANCELED`.

In [ ]:
# Submit a task just so we have something to cancel
cancel_task = TesTask(name="task-to-cancel").add_executor(
    image="ubuntu", command=["sleep", "300"]  # sleeps for 5 minutes
)
cancel_id = client.submit(cancel_task)
print(f"Submitted: {cancel_id}")

# Cancel it immediately
client.cancel(cancel_id)
print("Cancellation requested.")

# Check the state — may still be CANCELING for a moment
import time
time.sleep(2)
print(f"State after cancel: {client.state(cancel_id)}")

## 9. Handling failures

Things go wrong. This section shows how to detect and respond to the common failure modes.

### Task-level failures

A task can fail in two ways:
- **`EXECUTOR_ERROR`** — your command returned a non-zero exit code (e.g. the tool crashed, the input file was missing)
- **`SYSTEM_ERROR`** — something went wrong in the infrastructure (node failed, storage unavailable)

In [ ]:
# Deliberately failing task — exit code 1
failing_task = (
    TesTask(name="deliberate-failure")
    .add_executor(
        image="ubuntu",
        command=["bash", "-c", "echo 'Something went wrong'; exit 1"],
    )
)

fail_id = client.submit(failing_task)
print(f"Submitted failing task: {fail_id}")

tracker = TaskTracker(client, fail_id, poll_interval=10.0)
final = tracker.wait(timeout=300)

if final.state.is_failure():
    print(f"\nTask failed with state: {final.state}")
    print("\nDetailed report:")
    tracker.report()   # prints exit codes, stderr, and any system logs
else:
    print("Task completed.")

### Network and HTTP errors

If the TES server is unreachable or returns an error status, the client raises an `httpx.HTTPStatusError` or `httpx.ConnectError`. Wrap your calls in a try/except if you want to handle these gracefully:

In [ ]:
import httpx

try:
    task_id = client.submit(task)
    print(f"Submitted: {task_id}")

except httpx.ConnectError:
    print("Could not connect to the TES server — check the URL and your network.")

except httpx.HTTPStatusError as e:
    print(f"TES returned an error: {e.response.status_code} {e.response.text}")
    # 401 = authentication failed (check your credentials)
    # 403 = authorised but not permitted (check your Keycloak roles)
    # 400 = bad request (check your task JSON with task.submission_json())

### Timeout

`tracker.wait()` raises `TimeoutError` if the task hasn't finished within the specified number of seconds. You can catch this and decide what to do — cancel the task, log it for follow-up, or just report the last known state.

In [ ]:
long_task = (
    TesTask(name="potentially-long-task")
    .add_executor(image="ubuntu", command=["sleep", "3600"])  # 1 hour sleep
)
long_id = client.submit(long_task)

tracker = TaskTracker(client, long_id, poll_interval=10.0)

try:
    final = tracker.wait(timeout=30)  # only wait 30 seconds
    print(f"Finished: {final.state}")

except TimeoutError:
    current = client.state(long_id)
    print(f"Timed out waiting. Task {long_id[:8]}… is still in state: {current}")
    print("You can cancel it with: client.cancel(long_id)")

    # Clean up for this notebook demo
    client.cancel(long_id)
    print("Cancelled.")

---

## Summary

Here's the complete workflow in one place:

```python
from tes_client import ClientCredentialsAuth, TesClient, TesTask, TaskTracker

# 1. Authenticate
auth = ClientCredentialsAuth(
    base_url="https://keycloak.example.org",
    realm="my-realm",
    client_id="tes-service-account",
    client_secret="super-secret",
)

# 2. Connect
client = TesClient(tes_url="https://tes.example.org", token_manager=auth)

# 3. Build a task
task = (
    TesTask(name="my-analysis")
    .set_project_tag("DPUK_Project55")
    .add_input(url="s3://bucket/input.bam", path="/inputs/input.bam")
    .add_output(path="/outputs/result.txt", url="s3://bucket/result.txt")
    .set_resources(cpu_cores=4, ram_gb=8)
    .add_executor(image="ubuntu", command=["echo", "done"], stdout="/outputs/result.txt")
)

# 4. Check auth, submit, and track
auth.ensure_valid()
task_id = client.submit(task)
tracker = TaskTracker(client, task_id, poll_interval=15.0)
final = tracker.wait(timeout=3600)

if final.state.is_failure():
    tracker.report()
else:
    print("Done!")
```

---

**Next steps:**
- See `example_usage.py` in the repository root for a complete runnable script
- Read the `README.md` for the full API reference
- Ask your TES administrator for the server URL and your Keycloak credentials